In [2]:
import os
import numpy as np
import pandas as pd
import os
import json
import scipy.stats as stats
os.chdir(os.getcwd())

## CICIoV

In [45]:
def read_csv_files_in_folder(folder_path):
    """
    Reads CSV files from a folder and concatenates them into a single pandas DataFrame.

    Parameters:
    folder_path (str): Path to the folder containing the CSV files

    Returns:
    pandas.DataFrame: Concatenated DataFrame from all CSV files in the folder
    """
    # Get a list of all CSV files in the folder
    csv_files = [os.path.join(folder_path, f)
                 for f in os.listdir(folder_path) if f.endswith('.csv')]

    # Read each CSV file and append to a list of DataFrames
    dfs = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, low_memory=False)
        df['target'] = df['specific_class'].str.lower()
        dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [34]:
data = pd.read_parquet('./data/parquets/cicevse_network.parquet')

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 88 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            482309 non-null  int64  
 1   expiration_id                 482309 non-null  int64  
 2   src_ip                        482309 non-null  object 
 3   src_mac                       482309 non-null  object 
 4   src_oui                       482309 non-null  object 
 5   src_port                      482309 non-null  int64  
 6   dst_ip                        482309 non-null  object 
 7   dst_mac                       482309 non-null  object 
 8   dst_oui                       482309 non-null  object 
 9   dst_port                      482309 non-null  int64  
 10  protocol                      482309 non-null  int64  
 11  ip_version                    482309 non-null  int64  
 12  vlan_id                       482309 non-nul

In [38]:
data.head().iloc[:,10:]

,protocol,ip_version,vlan_id,tunnel_id,bidirectional_first_seen_ms,bidirectional_last_seen_ms,bidirectional_duration_ms,bidirectional_packets,bidirectional_bytes,src2dst_first_seen_ms,...,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,target,state
0,6,4,0,0,1703261991666,1703261991779,113,2,120,1703261991666,...,Unspecified,0,0,None,None,None,None,None,aggressive-scan,charging
1,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
2,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,System,1,1,None,None,None,None,None,aggressive-scan,charging
3,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
4,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,RPC,1,1,None,None,None,None,None,aggressive-scan,charging


In [42]:
def read_ciciov(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:,1:-3]
    y = df[target]
    return x, y

In [46]:
def read_cicevse(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:,14:-6]
    y = df[target]
    return x, y

In [46]:
def read_data(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1)
    y = df[target]
    return x, y

In [ ]:
def preprocess_data(X, y, features, batch_size):
    scaler = sklearn.preprocessing.RobustScaler()

    X_scaled = scaler.fit_transform(X[features].values)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32)
    n_features = X_scaled.shape[1]

    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=train_batch_size, shuffle=True)

    return X_tensor, y_tensor, dataloader

In [56]:
X, Y = read_ciciov('./data/parquets/ciciov.parquet', 'target')

In [57]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1408219 entries, 0 to 1408218
Data columns (total 8 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   DATA_0  1408219 non-null  int64
 1   DATA_1  1408219 non-null  int64
 2   DATA_2  1408219 non-null  int64
 3   DATA_3  1408219 non-null  int64
 4   DATA_4  1408219 non-null  int64
 5   DATA_5  1408219 non-null  int64
 6   DATA_6  1408219 non-null  int64
 7   DATA_7  1408219 non-null  int64
dtypes: int64(8)
memory usage: 86.0 MB


In [58]:
Y

0                  benign
1                  benign
2                  benign
3                  benign
4                  benign
                ...      
1408214    steering_wheel
1408215    steering_wheel
1408216    steering_wheel
1408217    steering_wheel
1408218    steering_wheel
Name: target, Length: 1408219, dtype: object

In [47]:
X, Y = read_cicevse('./data/parquets/cicevse_network.parquet', 'target')

In [48]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 67 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   bidirectional_first_seen_ms   482309 non-null  int64  
 1   bidirectional_last_seen_ms    482309 non-null  int64  
 2   bidirectional_duration_ms     482309 non-null  int64  
 3   bidirectional_packets         482309 non-null  int64  
 4   bidirectional_bytes           482309 non-null  int64  
 5   src2dst_first_seen_ms         482309 non-null  int64  
 6   src2dst_last_seen_ms          482309 non-null  int64  
 7   src2dst_duration_ms           482309 non-null  int64  
 8   src2dst_packets               482309 non-null  int64  
 9   src2dst_bytes                 482309 non-null  int64  
 10  dst2src_first_seen_ms         482309 non-null  int64  
 11  dst2src_last_seen_ms          482309 non-null  int64  
 12  dst2src_duration_ms           482309 non-nul

In [3]:
results = []
raw_results = {}
for file in os.listdir('results'):
    if file.endswith('.json'):
        file_path = os.path.join('results', file)
        with open(file_path, 'r') as f:
            data = json.load(f)
            results_arrays = {}
            results_arrays['Model'] = file
            results_arrays['Accuracy'] = np.mean(data['balanced_accuracy'])
            results_arrays['AUPRC'] = np.mean(data['val_AUPRC'])
            results_arrays['MCC'] = np.mean(data['val_MCC'])
            results_arrays['F1'] = np.mean(data['val_f1'])
            raw_results[file] = data
            results.append(results_arrays)
            # results_arrays['Loss'] = np.mean(data['val_loss'])

# # 3. Calculate mean of balanced_accuracy
# balanced_accuracies = [result['balanced_accuracy'] for result in results_arrays]
# mean_accuracy = np.mean(balanced_accuracies)

# print(f"Mean balanced accuracy: {mean_accuracy:.4f}")

In [5]:
baseline = 'no-polyak_euclidean-1_chebyshev-0_cosine-0_wasserstein-0.json'

In [ ]:

import numpy as np

def perform_ttest(group_a, group_b, alpha=0.05):
    mean_a = np.mean(group_a)
    mean_b = np.mean(group_b)
    std_a = np.std(group_a, ddof=1)  # ddof=1 for sample standard deviation
    std_b = np.std(group_b, ddof=1)
    
    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    
    print(f"Group A - Mean: {mean_a:.4f}, Std: {std_a:.4f}")
    print(f"Group B - Mean: {mean_b:.4f}, Std: {std_b:.4f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    if p_value < alpha:
        print("The difference is statistically significant (p < 0.05)")
    else:
        print("The difference is not statistically significant (p >= 0.05)")

perform_ttest(raw_results[baseline]['balanced_accuracy'], 
              raw_results['polyak_euclidean-1over4_chebyshev-1over4_cosine-1over4_wasserstein-1over4.json']['balanced_accuracy'])

Group A - Mean: 0.8460, Std: 0.0316
Group B - Mean: 0.8700, Std: 0.0274
t-statistic: -3.6303
p-value: 0.0005
The difference is statistically significant (p < 0.05)


In [7]:
pd.DataFrame(results).sort_values(by='Accuracy', ascending=False) # 200, att

,Model,Accuracy,AUPRC,MCC,F1
15,polyak_euclidean-1over2_chebyshev-0_cosine-1ov...,0.876787,0.465844,0.634678,0.662575
1,no-polyak_euclidean-1over3_chebyshev-1over3_co...,0.873976,0.479766,0.641218,0.672710
9,polyak_euclidean-0_chebyshev-1over3_cosine-1ov...,0.872501,0.451537,0.613864,0.649868
10,polyak_euclidean-1over3_chebyshev-1over3_cosin...,0.871558,0.464902,0.629037,0.659325
11,polyak_euclidean-1over4_chebyshev-1over4_cosin...,0.870034,0.415222,0.582885,0.617276
7,no-polyak_euclidean-1over2_chebyshev-0_cosine-...,0.870002,0.500339,0.656027,0.688587
0,no-polyak_euclidean-0_chebyshev-1over3_cosine-...,0.869473,0.459097,0.624010,0.656282
3,no-polyak_euclidean-1over4_chebyshev-1over4_co...,0.866381,0.448133,0.610248,0.646599
13,polyak_euclidean-1_chebyshev-0_cosine-0_wasser...,0.857176,0.401005,0.579437,0.605207
8,no-polyak_euclidean-1_chebyshev-0_cosine-0_was...,0.846050,0.391342,0.576802,0.598169


In [ ]:
200 